# ISS Group 24 Modeling

**Before running:**
1. Open the [shared Drive folder](https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing) and click **"Add shortcut to Drive"** (anywhere in your Drive is fine).
2. Select a GPU runtime: *Runtime → Change runtime type → T4 GPU*.
3. Run cells **in order** top-to-bottom. Each stage cell is idempotent — re-running a completed stage skips it automatically via checkpoint detection.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


In [ ]:
SHARED_FOLDER_LINK = "https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing"
REPO_URL        = "https://github.com/pewpewnor/iss_group_24.git"
REPO_LOCAL_PATH = "/content/iss_group_24_src"


In [ ]:
from google.colab import drive


def mount_drive() -> str:
    """Mount Google Drive and return the iss_group_24 project root path."""
    drive.mount("/content/drive")
    for candidate in [
        "/content/drive/Shareddrives/iss_group_24",
        "/content/drive/MyDrive/iss_group_24",
    ]:
        if Path(candidate).exists():
            print(f"Drive mounted. Project root: {candidate}")
            return candidate
    raise RuntimeError(
        "Shared folder 'iss_group_24' not found on this Drive.\n"
        f"Open the link and add a shortcut to your Drive: {SHARED_FOLDER_LINK}"
    )


def setup_repo() -> None:
    """Clone the repo at depth=1, or reset to origin/main if already cloned."""
    if Path(REPO_LOCAL_PATH).exists():
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "fetch", "origin"], check=True)
        subprocess.run(
            ["git", "-C", REPO_LOCAL_PATH, "reset", "--hard", "origin/main"],
            check=True,
        )
        print(f"Repo updated to origin/main: {REPO_LOCAL_PATH}")
    else:
        subprocess.run(
            ["git", "clone", "--depth=1", REPO_URL, REPO_LOCAL_PATH],
            check=True,
        )
        print(f"Repo cloned: {REPO_LOCAL_PATH}")
    sys.path.insert(0, REPO_LOCAL_PATH)


def install_deps() -> None:
    """Install packages not pre-installed on Colab (torch/torchvision are pre-installed)."""
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "pillow>=10.0", "scipy", "matplotlib>=3.7",
        ],
        check=True,
    )
    print("Dependencies ready")


In [ ]:
DRIVE_ROOT = mount_drive()
MANIFEST  = f"{DRIVE_ROOT}/dataset/cleaned/manifest.json"
DATA_ROOT = f"{DRIVE_ROOT}/dataset/cleaned"
MODEL_DIR = f"{DRIVE_ROOT}/model"

# Set CWD to the Drive project root so relative paths written by
# train.py (analysis/, manifest parent, etc.) land on the Drive.
os.chdir(DRIVE_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
setup_repo()

In [ ]:
install_deps()

In [ ]:
# requires setup_repo() to have run first
import aggregator
from modeling.train import train

def resume_from(ckpt: str | None = None):
    """Build the `resume` argument for train().

    - resume_from()                -> True (auto-resume from model/last.pt)
    - resume_from('stage1_2.pt')   -> '<MODEL_DIR>/stage1_2.pt' (force a specific ckpt)
    - resume_from('/abs/path.pt')  -> absolute path used as-is

    Use this in any train() call to override which checkpoint a stage starts
    from. Default behaviour (no argument) is unchanged: train() auto-resumes
    from model/last.pt if present, otherwise starts fresh.
    """
    if ckpt is None:
        return True
    if Path(ckpt).is_absolute():
        return str(ckpt)
    return f"{MODEL_DIR}/{ckpt}"


---
## Step 0 — Build Dataset Manifest

Copies and indexes all images from `dataset/original/` into `dataset/cleaned/`.  
**Run once.** Skip if `manifest.json` already exists.

In [ ]:
# Verify manifest AND staged files both exist; re-aggregate if either is missing.
train_dir = Path(DATA_ROOT) / "train"
staged_ok = (
    Path(MANIFEST).exists()
    and train_dir.is_dir()
    and any(train_dir.iterdir())
)
if staged_ok:
    print(f"Dataset already staged at {DATA_ROOT} — skipping aggregator.")
else:
    if Path(MANIFEST).exists():
        print("Manifest found but staged files are missing — re-running aggregator.")
    else:
        print("Running aggregator (copies images to dataset/cleaned/ — may take several minutes)…")
    aggregator.main()


---
## Training

Shared kwargs used by every stage cell. Each stage cell overrides only the relevant epoch count; the resume mechanism automatically skips already-completed stages.

In [ ]:
TRAIN_KWARGS = dict(
    manifest         = MANIFEST,
    data_root        = DATA_ROOT,
    out_dir          = MODEL_DIR,
    episodes_per_epoch = 2000,
    val_episodes     = 400,
    batch_size       = 16,
    num_workers      = 2,   # forkserver context avoids parent-RAM copy; safe on Colab T4
    resume           = True,  # default — auto-resume from model/last.pt. Override per cell with resume_from('stage1_2.pt') etc.
    contrastive_weight_p1 = 0.15,  # nt_xent was still high at epoch 10 with default 0.1
    # K-fold cross-validation for Phase 2. Pools HOTS+InsDet train+val,
    # rotates a held-out val fold per stage. Val is monitored only — no
    # early stopping, no checkpoint gating. Set to 0 to disable.
    phase2_cv_folds  = 4,
    # all stage epochs off by default — each cell turns on exactly one
    phase1_frozen_epochs  = 0,
    phase1_partial_epochs = 0,
    phase2_partial_epochs = 0,
    phase2_full_epochs    = 0,
)


---
## Phase 1 — General-Domain Pretraining on VizWiz

**Goal:** Train the few-shot matcher on a large, diverse set of everyday-object categories before it has ever seen the target domain (HOTS / InsDet products).

**Dataset:** VizWiz-FewShot base split — 100 categories, ~4 229 images of objects photographed with a smartphone (blurry, poorly lit, varied angles). This domain gap from ImageNet makes the pretrained backbone features imperfect, which is exactly what we want to fix.

**What gets learned:** The three matcher heads — SupportTokenizer, CrossAttentionHead, and DetHead — learn to compare a query image against a one-shot support example and produce a bounding-box prediction. The backbone starts frozen so the heads learn the episode format first, then the upper backbone blocks are partially thawed to adapt feature representations to phone-photo appearance.

**Outputs:** `stage1_1.pt` (frozen heads warmup), `stage1_2.pt` (partial backbone finetune), and a rolling `best.pt` / `last.pt`.

### Stage 1.1 — VizWiz base · backbone frozen · heads LR 1e-3

Warms up the matcher heads (SupportTokenizer + CrossAttentionHead + DetHead) on 100 diverse VizWiz categories using the rotation-synthesis episode scheme. Backbone stays frozen — no ImageNet features are modified yet.

In [ ]:
# To override which checkpoint this stage resumes from:
#   train(**{**TRAIN_KWARGS, "phase1_frozen_epochs": 15, "resume": resume_from("stage1_1.pt")})
train(**{**TRAIN_KWARGS, "phase1_frozen_epochs": 15})


### Stage 1.2 — VizWiz base · features[7:] @ 1e-5 · heads LR 5e-4

Partially unfreezes the upper MobileNetV3 blocks so the backbone can adapt its feature representations to the VizWiz scene domain while pretraining on diverse categories.

In [ ]:
# To override which checkpoint this stage resumes from:
#   train(**{**TRAIN_KWARGS, "phase1_partial_epochs": 40, "resume": resume_from("stage1_1.pt")})
train(**{**TRAIN_KWARGS, "phase1_partial_epochs": 40})


---
## Phase 2 — Target-Domain Transfer to HOTS + InsDet

Phase 2 takes the Phase-1-pretrained matcher (which learned generic few-shot object localization on 100 VizWiz categories) and adapts it to the actual evaluation domain: HOTS scene RGBs, InsDet product photos, and the 16 novel VizWiz document categories. This is a real distributional shift — backbone features, not just heads, need to move.

**Two stages:**

1. **Stage 2.1 — partial backbone unfreeze.** Upper MobileNetV3 blocks (`features[7:]`) join gradient flow at 3e-5; heads continue at 4e-4. This is where most of the target-domain adaptation happens. Hard-negative mining is active (ratio 0.5) so the model learns to reject visually similar distractors, `neg_prob` starts at 0.3 from epoch 1 (no warmup ramp — the heads were already exposed to negatives in Phase 1), and `presence_weight=2.0` doubles the presence-head loss term to fight the "always present" bias.
2. **Stage 2.2 — full polish.** Lower backbone (`features[0:7]`, the low-level ImageNet filters) joins for the first time at a very low 3e-6, with the upper blocks held at the same 3e-6 and heads dropped to 5e-5. Low LRs across the board — the goal is bounded fine-tuning, not aggressive movement.

The historical "frozen-backbone heads-only re-warmup" Phase-2 entry stage was removed: empirically it produced near-zero target-domain mAP because the heads needed the backbone to adapt in parallel, not in sequence. Stage 2.1 now is what used to be Stage 2.2; Stage 2.2 is what used to be Stage 2.3.

**Cross-validation (`phase2_cv_folds`).** With `phase2_cv_folds>0`, each Phase-2 stage cell runs that many folds end-to-end. HOTS+InsDet instances from the manifest's `train`+`val` splits are pooled and partitioned into K folds; each fold restarts from the post-Phase-1 weights, runs the stage's epoch budget on its training partition, and reports val metrics on the held-out fold. Val is monitored only — no early stopping, no checkpoint gating on val. `vizwiz_novel` (16 instances) stays fully in training across all folds. Per-fold histories: `analysis/cv/fold_<i>/history.json`; aggregated: `analysis/cv_results.json`. Set `phase2_cv_folds=0` to use the manifest's fixed train/val split instead.

**Resume / override.** By default each cell auto-resumes from `model/last.pt`. The Stage 2.1 cell explicitly overrides this to always start from `stage1_2.pt` (the post-Phase-1 weights) so re-running Phase 2 doesn't accidentally pick up a half-trained Phase-2 checkpoint. Stage 2.2 chains naturally from `last.pt` (which by then is the post-Stage-2.1 state). Both cells show the `resume_from(...)` pattern as a comment if you want to override further.

**Outputs:** `stage2_1.pt`, `stage2_2.pt`, and updated rolling `best.pt` / `last.pt`.

In [ ]:
TRAIN_KWARGS = dict(
    manifest         = MANIFEST,
    data_root        = DATA_ROOT,
    out_dir          = MODEL_DIR,
    episodes_per_epoch = 500,
    val_episodes     = 50,
    batch_size       = 16,
    num_workers      = 2,
    resume           = True,  # default — Stage 2.1 cell overrides this to stage1_2.pt explicitly
    contrastive_weight_p1 = 0.15,
    # K-fold cross-validation for Phase 2. 0 = use the manifest's fixed train/val split.
    phase2_cv_folds  = 0,
    # all stage epochs off by default — each cell turns on exactly one
    phase1_frozen_epochs  = 0,
    phase1_partial_epochs = 0,
    phase2_partial_epochs = 0,
    phase2_full_epochs    = 0,
)


### Stage 2.1 — partial unfreeze · features[7:] @ 3e-5 · heads LR 4e-4

Phase 2 entry stage. The upper MobileNetV3 blocks (`features[7:]`) join gradient flow at 3e-5 alongside the heads at 4e-4 — the heads were already warmed up by Phase 1 so a separate re-warmup is unnecessary. Hard-negative mining is on (ratio 0.5), `neg_prob` starts at 0.3 from epoch 1 (escalates to 0.4 at epoch 6), `presence_weight=2.0` to fight the presence-head bias toward 'always present'.

**This cell always resumes from `stage1_2.pt`** (the post-Phase-1 weights) so re-running it does not accidentally pick up a half-trained Phase-2 checkpoint.

In [ ]:
# Always start Stage 2.1 from the post-Phase-1 weights, not from last.pt.
train(**{**TRAIN_KWARGS, "phase2_partial_epochs": 15, "resume": resume_from("stage1_2.pt")})


### Stage 2.2 — full unfreeze · all backbone @ 3e-6 · heads LR 5e-5

Polish stage. Lower backbone (`features[0:7]`) joins gradient flow for the first time at 3e-6 to bound drift in the low-level ImageNet filters. Upper backbone is also at 3e-6; heads drop to 5e-5. Auto-resumes from `last.pt`, which by this point is the post-Stage-2.1 state.

In [ ]:
# To override which checkpoint this stage resumes from:
#   train(**{**TRAIN_KWARGS, "phase2_full_epochs": 8, "resume": resume_from("stage2_1.pt")})
train(**{**TRAIN_KWARGS, "phase2_full_epochs": 8})
